# Existence equation — panel logit

Whether a cross-pool arbitrage exists. Here the **onset** risk set (`gap_lag == 0`): given no gap
is open, does one appear at `t`?

$$\Pr\!\big(D_{p,t}=1 \mid X_{p,t-1}\big)=\Lambda(\eta_{p,t}),\qquad \Lambda(z)=\frac{1}{1+e^{-z}}$$

$$
\eta_{p,t}=\alpha_p
+\beta_2\,\log(\text{base\_fee}_t)+\beta_3\,\text{gas\_util}_{t-1}+\beta_4\,\log(1+\text{tip\_p90}_{t-1})
+\beta_5\,\overline{\log(1+\text{mev})}_{p,t-1}+\beta_6\,\overline{\text{nb\_swaps\_ewma}}_{p,t-1}
+\beta_7\,\log(\text{ewma\_vol}_t)+\beta_8\,\overline{\Delta\log L}_{p,t-1}
+\gamma_{h(t)}+\delta_{d(t)}+\eta\,\text{week}(t)
$$

`base_fee` and `ewma_vol` enter at `t`; every other regressor is lagged one block to be
predetermined. $\alpha_p$ are pool-pair fixed effects. All logic lives in `arblib.estimation`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

from arblib import estimation as est
from arblib.config import STUDY as S

panel = est.load_panel(S)
print("panel:", panel.shape)

panel: (4704, 28)


## Onset risk set

Rows with `gap_lag == 0`, dropping pairs with no `D` variation (uninformative under pair FE).

In [2]:
onset = est.build_risk_set(panel, quantile=0.2, condition="onset")

q20: dropping 15 pool pairs with no D variation (uninformative under pair FE): ['uniswap_1_vs_uniswap_4', 'uniswap_1_vs_uniswap_5', 'uniswap_2_vs_pancake_2', 'uniswap_2_vs_uniswap_3', 'uniswap_2_vs_uniswap_4', 'uniswap_2_vs_uniswap_5', 'uniswap_3_vs_pancake_1', 'uniswap_3_vs_pancake_2', 'uniswap_3_vs_uniswap_4', 'uniswap_3_vs_uniswap_5', 'uniswap_4_vs_pancake_1', 'uniswap_4_vs_pancake_2', 'uniswap_4_vs_uniswap_5', 'uniswap_5_vs_pancake_1', 'uniswap_5_vs_pancake_2']
onset: (1254, 17) | D mean: 0.0311 | pairs: 6


## Fit — logit & probit

Pool-pair fixed effects `C(pair)`, cluster-robust SEs by pair. Covariates are mean-centred so the
intercept is read at an average observation.

In [3]:
onset_c = est.center_continuous(onset, est.ONSET_TERMS)
res_logit  = est.fit_hazard_logit(onset_c, est.ONSET_TERMS, direction="logit")
res_probit = est.fit_hazard_logit(onset_c, est.ONSET_TERMS, direction="probit")
print(res_logit.summary())
print(res_probit.summary())

                           Logit Regression Results                           
Dep. Variable:                      D   No. Observations:                 1254
Model:                          Logit   Df Residuals:                     1241
Method:                           MLE   Df Model:                           12
Date:                Thu, 06 Aug 2026   Pseudo R-squ.:                  0.1488
Time:                        19:12:48   Log-Likelihood:                -147.89
converged:                       True   LL-Null:                       -173.74
Covariance Type:              cluster   LLR p-value:                 7.029e-07
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -5.2702      0.555     -9.503      0.000      -6.357      -4.183
C(pair)[T.uniswap_1_vs_pancake_1]     3.0631      0.890      3

## Average marginal effects

Logit coefficients are not probability changes, so report AMEs (logit and probit agree closely).

In [4]:
est.average_marginal_effects(res_logit, est.ONSET_TERMS)

,dy/dx,Std. Err.,z,Pr(>|z|),Conf. Int. Low,Cont. Int. Hi.
log_base_fee,-0.085736,0.018827,-4.554008,0.000005,-0.122636,-0.048837
gas_util_lag,0.032390,0.017351,1.866776,0.061933,-0.001617,0.066397
tip_p90_lag,0.021625,0.019740,1.095517,0.273290,-0.017064,0.060315
mev_lag,-0.010366,0.009017,-1.149514,0.250344,-0.028039,0.007308
freq_lag,0.046350,0.033167,1.397453,0.162277,-0.018657,0.111357
log_vol,-0.005288,0.035414,-0.149310,0.881309,-0.074698,0.064122
dlogL_lag,0.009984,0.007011,1.424136,0.154407,-0.003757,0.023726


## Multicollinearity check

VIFs (worry above ~10) and the covariate correlation matrix.

In [5]:
print(est.variance_inflation(onset, est.ONSET_TERMS))
onset[est.ONSET_TERMS].corr().round(3)

const           46196.330019
log_base_fee        1.106003
gas_util_lag        1.060344
tip_p90_lag         1.020564
mev_lag             1.674888
freq_lag            1.563005
log_vol             1.100584
dlogL_lag           1.001700
Name: VIF, dtype: float64


,log_base_fee,gas_util_lag,tip_p90_lag,mev_lag,freq_lag,log_vol,dlogL_lag
log_base_fee,1.000,0.202,0.040,0.194,0.150,0.159,0.014
gas_util_lag,0.202,1.000,0.120,-0.015,0.010,0.004,-0.025
tip_p90_lag,0.040,0.120,1.000,-0.023,0.038,-0.033,-0.027
mev_lag,0.194,-0.015,-0.023,1.000,0.589,0.252,-0.005
freq_lag,0.150,0.010,0.038,0.589,1.000,0.055,-0.004
log_vol,0.159,0.004,-0.033,0.252,0.055,1.000,0.004
dlogL_lag,0.014,-0.025,-0.027,-0.005,-0.004,0.004,1.000
